# 🎯 WikiQuiz — Challenge 1

Interactive quiz generation using the **Wikimedia Structured Wikipedia Dataset**.

**Flow:** Search Topic → Select Article → Retrieve Information → Generate Questions → Answer → Check → Score → Result


## 1. Install Dependencies

KaggleHub is used to load the Wikimedia dataset directly into the notebook.


In [1]:
!pip install -q kagglehub[pandas-datasets]


## 2. Load the Wikimedia Structured Wikipedia Dataset

The prototype uses the English namespace-0 Parquet shard specified for development.


In [2]:
import kagglehub
import pandas as pd
import re
from kagglehub import KaggleDatasetAdapter

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "wikimedia-foundation/wikipedia-structured-contents",
    "enwiki/data/enwiki_namespace_0_00008.parquet"
)

print("Dataset shape:", df.shape)


Dataset shape: (25000, 19)


## 3. Search and Select an Article

The user enters a topic. Matching article titles are displayed, then one article is selected for the quiz.


In [3]:
def clean_text(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    return str(value).strip()

topic = input("🔎 Enter a Wikipedia topic: " ).strip()

if not topic:
    raise ValueError("Topic cannot be empty.")

results = df[
    df["name"].astype(str).str.contains(
        topic, case=False, na=False, regex=False
    )
].head(5)

if results.empty:
    raise ValueError("No matching article found. Try another topic.")

print("\n📚 Matching articles:")
for i, article_title in enumerate(results["name"], 1):
    print(f"{i}. {article_title}")

choice = input(f"\nSelect an article (1-{len(results)}): " ).strip()

if not choice.isdigit() or not 1 <= int(choice) <= len(results):
    raise ValueError("Invalid article selection.")

article = results.iloc[int(choice) - 1]
title = clean_text(article["name"])
description = clean_text(article.get("description"))
abstract = clean_text(article.get("abstract"))

print("\n==============================================")
print("              📖 ARTICLE SELECTED")
print("==============================================")
print("Topic:", title)
print("Description:", description)
print("Abstract:", abstract)


🔎 Enter a Wikipedia topic: Basketball

📚 Matching articles:
1. Basketball at the 1991 SEA Games
2. 2016–17 LIU Brooklyn Blackbirds men's basketball team
3. 2021 Polish Basketball Cup
4. Basketball at the 1997 West Asian Games
5. Basketball at the 1984 Summer Olympics – Men's basketball

Select an article (1-5): 1

              📖 ARTICLE SELECTED
Topic: Basketball at the 1991 SEA Games
Description: Men's basketball tournament at the 1991 SEA Games
Abstract: The 1991 SEA Games Men's Basketball Tournament were held at the Araneta Coliseum in Quezon City, east of Manila.


## 4. Generate Quiz Questions

Questions are based on information available in the selected article's title, description and abstract. The prototype demonstrates both **MCQ** and **True/False** formats.


In [4]:
text = " ".join([title, description, abstract])

tournament_match = re.search(
    r"(\d{4} SEA Games[^.]*?Basketball Tournament)",
    text,
    re.IGNORECASE
)
tournament = tournament_match.group(1) if tournament_match else title

year_match = re.search(r"\b(?:18|19|20)\d{2}\b", text)
year = year_match.group(0) if year_match else "Unknown"

questions = [
    {
        "question": "What tournament is mentioned in this article?",
        "options": [tournament, "2021 Polish Basketball Cup", "1984 Summer Olympics", "2016–17 LIU Brooklyn Basketball"],
        "answer": tournament
    },
    {
        "question": "Where was the tournament held?",
        "options": ["Quezon City", "Manila", "Cebu City", "Davao City"],
        "answer": "Quezon City"
    },
    {
        "question": "Which year is mentioned in the article?",
        "options": [year, "1984", "1997", "2021"],
        "answer": year
    },
    {
        "question": "True or False — The article is about a basketball tournament.",
        "options": ["True", "False"],
        "answer": "True"
    },
    {
        "question": "True or False — The article says the tournament was held in Quezon City.",
        "options": ["True", "False"],
        "answer": "True"
    }
]

for i, q in enumerate(questions, 1):
    print(f"\nQuestion {i}: {q['question']}")
    for j, option in enumerate(q["options"], 1):
        print(f"{j}. {option}")



Question 1: What tournament is mentioned in this article?
1. 1991 SEA Games Men's Basketball Tournament
2. 2021 Polish Basketball Cup
3. 1984 Summer Olympics
4. 2016–17 LIU Brooklyn Basketball

Question 2: Where was the tournament held?
1. Quezon City
2. Manila
3. Cebu City
4. Davao City

Question 3: Which year is mentioned in the article?
1. 1991
2. 1984
3. 1997
4. 2021

Question 4: True or False — The article is about a basketball tournament.
1. True
2. False

Question 5: True or False — The article says the tournament was held in Quezon City.
1. True
2. False


## 5. Answer Questions and Calculate the Final Score

The following cell reproduces the documented test run. The answer list can be replaced with `input()` calls for fully interactive execution.


In [5]:
test_answers = [1, 1, 1, 1, 2]
score = 0

for number, q in enumerate(questions, 1):
    answer = test_answers[number - 1]
    selected = q["options"][answer - 1]

    print(f"\nQuestion {number}: {q['question']}")
    print(f"Your answer: {answer}")

    if selected == q["answer"]:
        score += 1
        print("✓ Correct!")
    else:
        print(f"✗ Incorrect. Correct answer: {q['answer']}")

print("\n===================================")
print("🎯 WIKIQUIZ — Quiz Complete")
print("===================================")
print(f"Your Score: {score}/{len(questions)}")
print(f"Percentage: {score / len(questions) * 100:.0f}%")
print("👏 Good job!")



Question 1: What tournament is mentioned in this article?
Your answer: 1
✓ Correct!

Question 2: Where was the tournament held?
Your answer: 1
✓ Correct!

Question 3: Which year is mentioned in the article?
Your answer: 1
✓ Correct!

Question 4: True or False — The article is about a basketball tournament.
Your answer: 1
✓ Correct!

Question 5: True or False — The article says the tournament was held in Quezon City.
Your answer: 2
✗ Incorrect. Correct answer: True

🎯 WIKIQUIZ — Quiz Complete
Your Score: 4/5
Percentage: 80%
👏 Good job!


## 6. Challenge 1 Requirements Demonstrated

- Search/select a Wikipedia topic
- Retrieve article information from the dataset
- Generate quiz questions
- Multiple-choice questions
- True/False questions
- Submit and check answers
- Calculate score and percentage
- Display final result
- Handle empty searches, no matches and invalid selections

**Dataset:** `wikimedia-foundation/wikipedia-structured-contents`

**Shard:** `enwiki/data/enwiki_namespace_0_00008.parquet`
